In [79]:
from attr import asdict, has
import inspect


class AutoSetterMixin:
    def __attrs_post_init__(self):
        self._apply_private_setters()

    def _apply_private_setters(self):
        for private_name in self.__attrs_attrs__:
            private_name = private_name.name

            if not private_name.startswith("_"):
                continue

            public_name = private_name[1:]

            prop = getattr(type(self), public_name, None)

            if isinstance(prop, property) and prop.fset is not None:
                setattr(self, public_name, getattr(self, private_name))

    def to_dict(self):
        data = {}

        for attr_field in self.__attrs_attrs__:
            name = attr_field.name
            value = getattr(self, name)

            if isinstance(value, AutoSetterMixin):
                data[name] = value.to_dict()
            elif has(value.__class__):
                data[name] = asdict(value)
            else:
                data[name] = value

        properties = {
            name: getattr(self, name)
            for name, value in inspect.getmembers(type(self))
            if isinstance(value, property)
        }

        data.update(properties)

        return data

In [103]:
from attr import define


@define
class RevenusData(AutoSetterMixin):
    nb_ch: int = 129
    nb_gu_ch: float = 1.7
    to: float = 0.8
    m_lin: float = 6

    _f_b_ponderation: float = 0.4
    _not_f_b_ponderation: float = 0.6

    f_b_ca_ht = 533
    not_f_b_ca_ht = 187

    f_b_ca_ttc = 587
    not_f_b_ca_ttc = 224

    f_b_marge: float = 2.6
    not_f_b_marge: float = 1.45

    _nb_ventes_mensuelles: int = 231
    _cl_acheteurs_mois: int = 0.0432


    @property
    def f_b_ponderation(self):
        return self._f_b_ponderation

    @f_b_ponderation.setter
    def f_b_ponderation(self, value):
        if (0 <= value <= 1):
            self._f_b_ponderation = value
            self._not_f_b_ponderation = 1 - value

    @property
    def not_f_b_ponderation(self):
        return self._not_f_b_ponderation

    @not_f_b_ponderation.setter
    def not_f_b_ponderation(self, value):
        if (0 <= value <= 1):
            self._not_f_b_ponderation = value
            self._f_b_ponderation = 1 - value

    @property
    def marge_ponderee(self):
        return (
            self.f_b_ponderation * self.f_b_marge
            + self.not_f_b_ponderation * self.not_f_b_marge
        )

    @property
    def ch_occ(self):
        return self.nb_ch * self.to

    @property
    def cl_heb_jour(self):
        return self.nb_ch * self.nb_gu_ch * self.to

    @property
    def cl_heb_mois(self):
        return self.cl_heb_jour * 30.5

    @property
    def cl_acheteurs_mois(self):
        return self._cl_acheteurs_mois
    

    @cl_acheteurs_mois.setter
    def cl_acheteurs_mois(self, value):
        self._cl_acheteurs_mois = value
        self._nb_ventes_mensuelles = self._cl_acheteurs_mois * self.cl_heb_mois

    @property
    def nb_ventes_mensuelles(self):
        return self._nb_ventes_mensuelles
    

    @nb_ventes_mensuelles.setter
    def nb_ventes_mensuelles(self, value):
        print(value)
        self._nb_ventes_mensuelles = value
        self._cl_acheteurs_mois = self._nb_ventes_mensuelles / self.cl_heb_mois


    

In [104]:
from attr import define


@define
class SimRevenusData(RevenusData):
    _pilote_revenus_data : RevenusData = None
    
    @property
    def pilote_revenus_data(self):
        return self._pilote_revenus_data
    

    @pilote_revenus_data.setter
    def pilote_revenus_data(self, value):
        self._pilote_revenus_data = value
        if(self._pilote_revenus_data is not None):
            print(self._pilote_revenus_data.cl_acheteurs_mois * self.cl_heb_mois)
            self.nb_ventes_mensuelles = self._pilote_revenus_data.cl_acheteurs_mois * self.cl_heb_mois


srd = SimRevenusData(
    nb_ch=100, nb_gu_ch=1.5, to=0.8, m_lin=3, f_b_ponderation=0.7, not_f_b_ponderation=0.3,
    pilote_revenus_data = RevenusData()
)

srd.nb_ventes_mensuelles = 158
d = srd.to_dict()
d.pop("_pilote_revenus_data")


231
231
158.00273597811216
158.00273597811216
158


{'nb_ch': 129,
 'nb_gu_ch': 1.7,
 'to': 0.8,
 'm_lin': 6,
 '_f_b_ponderation': 0.4,
 '_not_f_b_ponderation': 0.6,
 'f_b_marge': 2.6,
 'not_f_b_marge': 1.45,
 '_nb_ventes_mensuelles': 231.0,
 '_cl_acheteurs_mois': 0.04317014644210715,
 'ch_occ': 103.2,
 'cl_acheteurs_mois': 0.04317014644210715,
 'cl_heb_jour': 175.44,
 'cl_heb_mois': 5350.92,
 'f_b_ponderation': 0.4,
 'marge_ponderee': 1.9100000000000001,
 'nb_ventes_mensuelles': 231.0,
 'not_f_b_ponderation': 0.6}